# Solutions: Notebook 04 - Fixed Effects without Location Shift (Penalty Method)

Complete solutions for all exercises in Notebook 04.

In [ ]:
# Standard libraries
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")

# Statistical libraries

# PanelBox imports
from panelbox.core.panel_data import PanelData
from panelbox.models.quantile import CanayTwoStep, FixedEffectsQuantile, PooledQuantile

# Visualization configuration
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("husl")
plt.rcParams["figure.figsize"] = (12, 7)
plt.rcParams["figure.dpi"] = 100
plt.rcParams["font.size"] = 11
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12
pd.set_option("display.max_columns", None)
pd.set_option("display.precision", 4)

# Reproducibility
np.random.seed(42)

# Define paths (relative to solutions/)
BASE_DIR = Path("..")
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "outputs"
PLOTS_DIR = OUTPUT_DIR / "plots"
RESULTS_DIR = OUTPUT_DIR / "results"

PLOTS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Setup complete!")

In [ ]:
# Load data (shared setup for all exercises)
data = pd.read_csv(DATA_DIR / "financial_returns.csv")
panel = PanelData(data, entity_col="firm_id", time_col="month")
formula = "returns ~ size + book_to_market + momentum"

# Prepare arrays for PooledQuantile
y = data["returns"].values
X = np.column_stack(
    [
        np.ones(len(data)),
        data["size"].values,
        data["book_to_market"].values,
        data["momentum"].values,
    ]
)
entity_id = data["firm_id"].values
var_names = ["const", "size", "book_to_market", "momentum"]

# Pre-fit penalty results at key quantiles for use in exercises
print("Pre-fitting penalty models for exercises...")
penalty_results = {}
for tau in [0.1, 0.5, 0.9]:
    model = FixedEffectsQuantile(panel, formula=formula, tau=tau, lambda_fe="auto")
    penalty_results[tau] = model.fit(cv_folds=5, verbose=False)
    res = penalty_results[tau].results[tau]
    print(
        f"  \u03c4={tau}: \u03bb={res.lambda_fe:.4f}, "
        f"# Zero FE = {int(np.sum(np.abs(res.fixed_effects) < 1e-6))}"
    )

# Build firm_profiles for exercises 4, 5, 6
all_firms = data["firm_id"].unique()
firm_profiles = pd.DataFrame(index=all_firms)
for tau in [0.1, 0.5, 0.9]:
    res = penalty_results[tau].results[tau]
    model_obj = res.model
    alphas = []
    for firm in all_firms:
        if firm in model_obj.entity_map:
            idx = model_obj.entity_map[firm]
            alphas.append(res.fixed_effects[idx])
        else:
            alphas.append(0.0)
    firm_profiles[f"alpha_{int(100 * tau)}"] = alphas

firm_profiles["fe_spread"] = firm_profiles["alpha_90"] - firm_profiles["alpha_10"]

print(f"\nData loaded: {data.shape}")
print(f"Panel: {panel}")
print(f"Firm profiles computed for {len(all_firms)} firms")

---

## Exercise 1: L1 vs L2 Penalty (Easy)

**Question**: Why does the Koenker penalty use L1 (Lasso) instead of L2 (Ridge) for the fixed effects?

In [ ]:
# Exercise 1 Solution

print("=" * 70)
print("SOLUTION: L1 vs L2 Penalty for Fixed Effects")
print("=" * 70)

print("""
THEORETICAL COMPARISON:

L1 Penalty (Lasso):  lambda * sum|alpha_i(tau)|
L2 Penalty (Ridge):  lambda * sum(alpha_i(tau)^2)

Property              L1 (Lasso)              L2 (Ridge)
-------------------------------------------------------------
Sparsity              Sets some alpha_i = 0   Shrinks but never = 0
Bias                  Less bias at large tau   More bias overall
Computation           LP (linear program)     QP (quadratic program)
Many entities (N>>T)  Handles well (sparsity) May oversmooth
Interpretability      Identifies "zero-FE"    All FE nonzero
                      firms automatically

WHY L1 IS PREFERRED:

1. SPARSITY: In many panels, some entities have negligible fixed effects.
   L1 sets these exactly to zero, providing automatic model selection.
   L2 keeps all FE nonzero, adding noise.

2. COMPUTATIONAL TRACTABILITY: The L1-penalized quantile objective
   can be reformulated as a linear program (LP), which is solvable
   by simplex or interior point methods. L2 + quantile loss gives a
   harder quadratic program.

3. CONSISTENCY: Koenker (2004) shows that L1 penalty with
   lambda = lambda(N,T) -> 0 as N,T -> infinity gives consistent
   beta(tau) estimates, even with incidental parameters.

4. INTERPRETATION: Firms with alpha_i(tau) = 0 are "average" at
   that quantile. This is economically meaningful — they have no
   firm-specific deviation from the population quantile.
""")

In [ ]:
# Numerical demonstration: sparsity of L1

print("Numerical Demonstration: Sparsity as lambda increases")
print("=" * 60)

# Fit with different lambda values and count zero FE
lambda_test = [0.01, 0.1, 1.0, 5.0, 10.0, 50.0, 100.0]
tau = 0.5

sparsity_results = []
for lam in lambda_test:
    model = FixedEffectsQuantile(panel, formula=formula, tau=tau, lambda_fe=lam)
    result = model.fit(verbose=False)
    res = result.results[tau]

    n_zero = int(np.sum(np.abs(res.fixed_effects) < 1e-6))
    n_total = len(res.fixed_effects)
    fe_std = np.std(res.fixed_effects)

    sparsity_results.append(
        {
            "lambda": lam,
            "n_zero": n_zero,
            "n_total": n_total,
            "pct_zero": 100 * n_zero / n_total,
            "fe_std": fe_std,
        }
    )
    print(
        f"\u03bb={lam:6.2f}: {n_zero:3d}/{n_total} zero FE "
        f"({100 * n_zero / n_total:5.1f}%), FE std={fe_std:.4f}"
    )

# Visualize sparsity progression
sp_df = pd.DataFrame(sparsity_results)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: % zero FE vs lambda
ax1.semilogx(sp_df["lambda"], sp_df["pct_zero"], "o-", linewidth=2, markersize=8, color="steelblue")
ax1.set_xlabel("Penalty Parameter (\u03bb)", fontsize=12)
ax1.set_ylabel("% Zero Fixed Effects", fontsize=12)
ax1.set_title("L1 Sparsity: More Zeros as \u03bb Increases", fontsize=13, fontweight="bold")
ax1.grid(alpha=0.3)
ax1.set_ylim(-5, 105)

# Plot 2: FE standard deviation vs lambda
ax2.semilogx(sp_df["lambda"], sp_df["fe_std"], "s-", linewidth=2, markersize=8, color="darkred")
ax2.set_xlabel("Penalty Parameter (\u03bb)", fontsize=12)
ax2.set_ylabel("FE Standard Deviation", fontsize=12)
ax2.set_title("L1 Shrinkage: FE Dispersion Decreases", fontsize=13, fontweight="bold")
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(
    "\nKey observation: L1 penalty produces EXACT zeros (sparsity),\n"
    "not just small values. This is the fundamental difference from L2."
)

---

## Exercise 2: Manual Cross-Validation (Easy)

**Task**: Implement a manual entity-based cross-validation loop for $\lambda$ selection.

In [ ]:
# Exercise 2 Solution: Manual entity-based CV

print("=" * 70)
print("MANUAL ENTITY-BASED CROSS-VALIDATION")
print("=" * 70)

tau_cv = 0.5
lambda_grid = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]
K = 3  # Number of folds

# Step 1: Split firms into K folds
unique_firms = data["firm_id"].unique()
np.random.seed(42)
np.random.shuffle(unique_firms)
firm_folds = np.array_split(unique_firms, K)

print(f"\nNumber of firms: {len(unique_firms)}")
for k in range(K):
    print(f"  Fold {k + 1}: {len(firm_folds[k])} firms")

# Step 2: CV loop
cv_scores = {lam: [] for lam in lambda_grid}

print(f"\nRunning {K}-fold CV for {len(lambda_grid)} lambda values...")
for fold_idx in range(K):
    # Held-out firms
    test_firms = set(firm_folds[fold_idx])

    # Split data
    train_mask = ~data["firm_id"].isin(test_firms)
    test_mask = data["firm_id"].isin(test_firms)
    train_data = data[train_mask].copy()
    test_data = data[test_mask].copy()

    train_panel = PanelData(train_data, entity_col="firm_id", time_col="month")

    print(f"\n  Fold {fold_idx + 1}: train={len(train_data)}, test={len(test_data)}")

    for lam in lambda_grid:
        # Fit on training set
        model = FixedEffectsQuantile(train_panel, formula=formula, tau=tau_cv, lambda_fe=lam)
        result = model.fit(verbose=False)
        res = result.results[tau_cv]

        # Predict on test set (no FE for unseen firms)
        y_test = test_data["returns"].values
        X_test = np.column_stack(
            [
                np.ones(len(test_data)),
                test_data["size"].values,
                test_data["book_to_market"].values,
                test_data["momentum"].values,
            ]
        )

        # Predict using only beta (FE unknown for test firms)
        y_pred = X_test @ res.params
        residuals = y_test - y_pred

        # Check loss on test set
        check_loss = np.mean(residuals * (tau_cv - (residuals < 0).astype(float)))
        cv_scores[lam].append(check_loss)

# Step 3: Average CV scores
print(f"\n{'\u03bb':>10} {'CV Score':>12} {'Std':>10}")
print("-" * 35)

mean_scores = {}
for lam in lambda_grid:
    mean_s = np.mean(cv_scores[lam])
    std_s = np.std(cv_scores[lam])
    mean_scores[lam] = mean_s
    print(f"{lam:10.3f} {mean_s:12.6f} {std_s:10.6f}")

best_lambda_manual = min(mean_scores, key=mean_scores.get)
print(f"\nBest \u03bb (manual CV): {best_lambda_manual}")

In [ ]:
# Compare manual CV with automatic CV
print("\nComparing manual vs automatic CV:")
print(f"  Manual CV best \u03bb: {best_lambda_manual}")

auto_model = FixedEffectsQuantile(panel, formula=formula, tau=tau_cv, lambda_fe="auto")
auto_result = auto_model.fit(cv_folds=3, verbose=False)
auto_lambda = auto_result.results[tau_cv].lambda_fe
print(f"  Auto CV best \u03bb:   {auto_lambda:.4f}")

print("\nNote: Differences are expected because:")
print("  1. Automatic CV uses a finer lambda grid (20 values)")
print("  2. Random fold assignments may differ")
print("  3. Automatic CV uses log-spaced grid, not manual values")

---

## Exercise 3: Lambda Extremes (Medium)

**Task**: Show that $\lambda = 0$ gives unconstrained FE and $\lambda \to \infty$ gives pooled QR.

In [ ]:
# Exercise 3 Solution: Lambda extremes

print("=" * 70)
print("LAMBDA EXTREMES: UNCONSTRAINED FE vs POOLED QR")
print("=" * 70)

tau = 0.5

# 1. Near-zero lambda (unconstrained FE)
print("\n1. Near-zero \u03bb (unconstrained fixed effects):")
model_small = FixedEffectsQuantile(panel, formula=formula, tau=tau, lambda_fe=0.001)
res_small = model_small.fit(verbose=False).results[tau]
print("   \u03bb = 0.001")
print(f"   FE std:   {np.std(res_small.fixed_effects):.4f}")
print(
    f"   FE range: [{np.min(res_small.fixed_effects):.4f}, {np.max(res_small.fixed_effects):.4f}]"
)
print(f"   # Zero:   {int(np.sum(np.abs(res_small.fixed_effects) < 1e-6))}")
print(f"   Coefficients: {res_small.params.round(4)}")

# 2. Very large lambda (all FE -> 0 = pooled QR)
print("\n2. Very large \u03bb (\u03b1_i -> 0, approaches pooled QR):")
model_large = FixedEffectsQuantile(panel, formula=formula, tau=tau, lambda_fe=1000.0)
res_large = model_large.fit(verbose=False).results[tau]
print("   \u03bb = 1000")
print(f"   FE std:   {np.std(res_large.fixed_effects):.6f}")
print(
    f"   FE range: [{np.min(res_large.fixed_effects):.6f}, {np.max(res_large.fixed_effects):.6f}]"
)
print(f"   # Zero:   {int(np.sum(np.abs(res_large.fixed_effects) < 1e-6))}")
print(f"   Coefficients: {res_large.params.round(4)}")

# 3. Pooled QR (no FE at all)
print("\n3. Pooled QR (no fixed effects):")
pooled_model = PooledQuantile(y, X, entity_id=entity_id, quantiles=tau)
pooled_result = pooled_model.fit(se_type="cluster")
print(f"   Coefficients: {pooled_result.params.ravel().round(4)}")

In [ ]:
# Visualize the three regimes
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: FE distribution for small lambda
axes[0].hist(res_small.fixed_effects, bins=30, edgecolor="black", alpha=0.7, color="#FF5722")
axes[0].axvline(0, color="black", linestyle="--", linewidth=1.5)
axes[0].set_xlabel("Fixed Effect", fontsize=11)
axes[0].set_ylabel("Frequency", fontsize=11)
axes[0].set_title(
    f"\u03bb = 0.001 (Unconstrained)\nStd = {np.std(res_small.fixed_effects):.4f}",
    fontsize=12,
    fontweight="bold",
)
axes[0].grid(alpha=0.3)

# Plot 2: FE distribution for large lambda
axes[1].hist(res_large.fixed_effects, bins=30, edgecolor="black", alpha=0.7, color="#4CAF50")
axes[1].axvline(0, color="black", linestyle="--", linewidth=1.5)
axes[1].set_xlabel("Fixed Effect", fontsize=11)
axes[1].set_ylabel("Frequency", fontsize=11)
axes[1].set_title(
    f"\u03bb = 1000 (Nearly Pooled)\nStd = {np.std(res_large.fixed_effects):.6f}",
    fontsize=12,
    fontweight="bold",
)
axes[1].grid(alpha=0.3)

# Plot 3: Coefficient comparison
small_params = res_small.params
large_params = res_large.params
pooled_params = pooled_result.params.ravel()

x_pos = np.arange(len(var_names))
width = 0.25

axes[2].bar(x_pos - width, small_params, width, label="\u03bb=0.001", color="#FF5722", alpha=0.8)
axes[2].bar(x_pos, large_params, width, label="\u03bb=1000", color="#4CAF50", alpha=0.8)
axes[2].bar(x_pos + width, pooled_params, width, label="Pooled QR", color="#2196F3", alpha=0.8)
axes[2].set_xticks(x_pos)
axes[2].set_xticklabels(var_names, rotation=15)
axes[2].set_ylabel("Coefficient", fontsize=11)
axes[2].set_title("Slope Coefficients (\u03c4=0.5)", fontsize=12, fontweight="bold")
axes[2].legend(fontsize=10)
axes[2].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

print("\nConclusion:")
print("  \u03bb -> 0:   Large FE variance, FE freely estimated (risk of overfitting)")
print("  \u03bb -> inf: All FE shrunk to 0, slopes converge to pooled QR")
print("  The penalty parameter interpolates between these extremes.")

---

## Exercise 4: FE Scatter Plot (Medium)

**Task**: Create a scatter plot of $\hat{\alpha}_i(0.1)$ vs $\hat{\alpha}_i(0.9)$ for all firms.

In [ ]:
# Exercise 4 Solution: Full scatter plot

print("=" * 70)
print("FE SCATTER: \u03b1(0.1) vs \u03b1(0.9) FOR ALL FIRMS")
print("=" * 70)

alpha_10 = firm_profiles["alpha_10"].values
alpha_90 = firm_profiles["alpha_90"].values

# Compute correlation
corr = np.corrcoef(alpha_10, alpha_90)[0, 1]
print(f"\nCorrelation \u03b1(0.1) vs \u03b1(0.9): {corr:.4f}")

# Classify firms by risk type
q25 = firm_profiles["fe_spread"].quantile(0.25)
q75 = firm_profiles["fe_spread"].quantile(0.75)

sym_mask = firm_profiles["fe_spread"] <= q25
asym_mask = firm_profiles["fe_spread"] >= q75
mid_mask = ~sym_mask & ~asym_mask

# Create scatter plot
fig, ax = plt.subplots(figsize=(10, 8))

ax.scatter(
    alpha_10[mid_mask], alpha_90[mid_mask], c="#9E9E9E", alpha=0.5, s=40, label="Medium spread"
)
ax.scatter(
    alpha_10[sym_mask],
    alpha_90[sym_mask],
    c="#2196F3",
    alpha=0.7,
    s=50,
    label="Low spread (Symmetric)",
)
ax.scatter(
    alpha_10[asym_mask],
    alpha_90[asym_mask],
    c="#FF5722",
    alpha=0.7,
    s=50,
    label="High spread (Asymmetric)",
)

# 45-degree line (location shift line)
lim = max(np.abs(alpha_10).max(), np.abs(alpha_90).max()) * 1.2
ax.plot(
    [-lim, lim], [-lim, lim], "k--", alpha=0.5, linewidth=2, label="45\u00b0 line (location shift)"
)

# Zero lines
ax.axhline(0, color="gray", linestyle=":", alpha=0.3)
ax.axvline(0, color="gray", linestyle=":", alpha=0.3)

ax.set_xlabel("\u03b1\u0302\u1d62(0.1) — Left tail FE", fontsize=12)
ax.set_ylabel("\u03b1\u0302\u1d62(0.9) — Right tail FE", fontsize=12)
ax.set_title(
    f"Fixed Effects: Left Tail vs Right Tail\nCorrelation = {corr:.3f}",
    fontsize=14,
    fontweight="bold",
)
ax.legend(fontsize=10, loc="upper left")
ax.grid(alpha=0.3)
ax.set_aspect("equal", adjustable="datalim")

plt.tight_layout()
plt.show()

In [ ]:
# Interpretation
print("\nINTERPRETATION:")
print("=" * 60)

print(f"\n1. Correlation \u03b1(0.1) vs \u03b1(0.9) = {corr:.3f}")
if corr > 0.9:
    print("   -> High correlation suggests location shift may hold.")
    print("   -> Canay estimator would be appropriate.")
elif corr > 0.5:
    print(
        "   -> Moderate correlation: some location shift component,\n"
        "      but significant quantile-varying heterogeneity."
    )
else:
    print(
        "   -> Low correlation: fixed effects vary substantially\n"
        "      across quantiles. Location shift is a poor assumption."
    )

# Count firms far from 45-degree line
spread = firm_profiles["fe_spread"].values
far_firms = np.sum(np.abs(spread) > 0.2)
print(f"\n2. Firms far from 45\u00b0 line (|spread| > 0.2): {far_firms}/{len(all_firms)}")
print("   These firms have meaningfully different FE at tails vs center.")

print(f"\n3. Correlation sign: {'Positive' if corr > 0 else 'Negative'}")
if corr > 0:
    print(
        "   -> Firms with positive left-tail FE tend to have positive\n"
        "      right-tail FE. The location component dominates."
    )
else:
    print(
        "   -> Firms with positive left-tail FE tend to have negative\n"
        '      right-tail FE. This suggests "mean-reverting" risk profiles.'
    )

---

## Exercise 5: Simulation — Canay Bias Under Location Shift Violation (Hard)

**Task**: Simulate data where the location shift assumption is violated, then show that Canay is biased while Penalty is not.

In [ ]:
# Exercise 5 Solution: Simulation study

print("=" * 70)
print("SIMULATION: CANAY BIAS UNDER LOCATION SHIFT VIOLATION")
print("=" * 70)

# DGP: y_it = alpha_i(tau) + beta * x_it + epsilon_it
# where alpha_i(tau) = mu_i + sigma_i * F^{-1}(tau)
#       mu_i ~ N(0, 1)         -> location component
#       sigma_i ~ U(0.5, 2)    -> scale component (violates location shift!)
#       beta = 1.0             -> true slope
#       epsilon_it ~ N(0, 1)

np.random.seed(42)
N_sim = 50  # firms
T_sim = 20  # periods
beta_true = 1.0

# Generate entity-specific parameters
mu_i = np.random.normal(0, 1, N_sim)
sigma_i = np.random.uniform(0.5, 2.0, N_sim)  # Scale varies -> violates location shift

# Generate panel data
rows = []
for i in range(N_sim):
    for t in range(T_sim):
        x_it = np.random.normal(0, 1)
        eps_it = np.random.normal(0, sigma_i[i])  # Heteroskedastic errors
        y_it = mu_i[i] + beta_true * x_it + eps_it
        rows.append({"id": i + 1, "time": t + 1, "y": y_it, "x": x_it})

sim_data = pd.DataFrame(rows)
sim_panel = PanelData(sim_data, entity_col="id", time_col="time")
sim_formula = "y ~ x"

print(f"\nSimulated data: N={N_sim} firms, T={T_sim} periods, {len(sim_data)} observations")
print(f"True beta = {beta_true}")
print("mu_i ~ N(0,1), sigma_i ~ U(0.5, 2.0)")
print("This DGP VIOLATES location shift (heteroskedastic FE).")

In [ ]:
# Estimate Canay and Penalty at different quantiles
sim_taus = [0.1, 0.25, 0.5, 0.75, 0.9]

canay_sim = {}
penalty_sim = {}

# Prepare arrays for pooled QR
y_sim = sim_data["y"].values
X_sim = np.column_stack([np.ones(len(sim_data)), sim_data["x"].values])
eid_sim = sim_data["id"].values

print("Estimating models...")
for tau in sim_taus:
    # Canay
    canay_model = CanayTwoStep(sim_panel, formula=sim_formula, tau=tau)
    canay_sim[tau] = canay_model.fit(verbose=False)

    # Penalty
    penalty_model = FixedEffectsQuantile(sim_panel, formula=sim_formula, tau=tau, lambda_fe="auto")
    penalty_sim[tau] = penalty_model.fit(cv_folds=3, verbose=False)

# Compare estimates
print(f"\n{'tau':<6} {'True':>8} {'Canay':>10} {'Bias':>8} {'Penalty':>10} {'Bias':>8}")
print("-" * 55)

for tau in sim_taus:
    canay_beta = canay_sim[tau].results[tau].params[1]  # x coefficient
    penalty_beta = penalty_sim[tau].results[tau].params[1]  # x coefficient
    canay_bias = canay_beta - beta_true
    penalty_bias = penalty_beta - beta_true

    print(
        f"{tau:<6.2f} {beta_true:8.4f} {canay_beta:10.4f} "
        f"{canay_bias:8.4f} {penalty_beta:10.4f} {penalty_bias:8.4f}"
    )

In [ ]:
# Visualize the bias comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Coefficient estimates
canay_betas = [canay_sim[tau].results[tau].params[1] for tau in sim_taus]
penalty_betas = [penalty_sim[tau].results[tau].params[1] for tau in sim_taus]

axes[0].plot(sim_taus, canay_betas, "o-", linewidth=2, markersize=8, label="Canay", color="#F44336")
axes[0].plot(
    sim_taus, penalty_betas, "s-", linewidth=2, markersize=8, label="Penalty", color="#4CAF50"
)
axes[0].axhline(
    beta_true, color="black", linestyle="--", linewidth=2, label=f"True \u03b2 = {beta_true}"
)
axes[0].set_xlabel("Quantile (\u03c4)", fontsize=12)
axes[0].set_ylabel("Estimated \u03b2", fontsize=12)
axes[0].set_title("Slope Estimates: Canay vs Penalty", fontsize=13, fontweight="bold")
axes[0].legend(fontsize=10)
axes[0].grid(alpha=0.3)

# Plot 2: Bias magnitude
canay_biases = [canay_sim[tau].results[tau].params[1] - beta_true for tau in sim_taus]
penalty_biases = [penalty_sim[tau].results[tau].params[1] - beta_true for tau in sim_taus]

x_pos = np.arange(len(sim_taus))
width = 0.35

axes[1].bar(
    x_pos - width / 2, np.abs(canay_biases), width, label="|Canay bias|", color="#F44336", alpha=0.8
)
axes[1].bar(
    x_pos + width / 2,
    np.abs(penalty_biases),
    width,
    label="|Penalty bias|",
    color="#4CAF50",
    alpha=0.8,
)
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels([f"\u03c4={t}" for t in sim_taus])
axes[1].set_ylabel("|Bias|", fontsize=12)
axes[1].set_title("Absolute Bias Comparison", fontsize=13, fontweight="bold")
axes[1].legend(fontsize=10)
axes[1].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

print("\nConclusion:")
print("  Under location shift VIOLATION (heteroskedastic FE):")
print("  - Canay is biased, especially at tail quantiles")
print("  - Penalty method is less biased because it allows alpha_i(tau) to vary")
print("  - The bias pattern reflects Canay imposing incorrect constant-FE assumption")

---

## Exercise 6: Application — Wage Panel (Hard)

**Task**: Use the Card education dataset to study whether worker fixed effects (ability) vary across the wage distribution.

In [ ]:
# Exercise 6 Solution: Wage panel application

print("=" * 70)
print("APPLICATION: WORKER ABILITY HETEROGENEITY ACROSS WAGE QUANTILES")
print("=" * 70)

# Load Card education dataset
wage_data = pd.read_csv(DATA_DIR / "card_education.csv")
print(f"\nDataset shape: {wage_data.shape}")
print(f"Variables: {wage_data.columns.tolist()}")
print(f"Workers: {wage_data['id'].nunique()}")
print(f"Years: {wage_data['year'].nunique()}")

wage_panel = PanelData(wage_data, entity_col="id", time_col="year")

# Use time-varying regressors only
# (educ, female, black are time-invariant and absorbed by FE)
wage_formula = "lwage ~ exper + married + union"

print(f"\nFormula: {wage_formula}")
print("Time-invariant variables (educ, female, black) absorbed by FE.")

print("\nDescriptive statistics:")
display(wage_data[["lwage", "exper", "married", "union"]].describe())

In [ ]:
# Estimate penalty FE-QR at multiple quantiles
wage_taus = [0.1, 0.25, 0.5, 0.75, 0.9]

wage_penalty = {}
wage_canay = {}

print("Estimating penalty FE-QR...")
for tau in wage_taus:
    # Penalty
    model_p = FixedEffectsQuantile(wage_panel, formula=wage_formula, tau=tau, lambda_fe="auto")
    wage_penalty[tau] = model_p.fit(cv_folds=3, verbose=False)

    # Canay (for comparison)
    model_c = CanayTwoStep(wage_panel, formula=wage_formula, tau=tau)
    wage_canay[tau] = model_c.fit(verbose=False)

    res_p = wage_penalty[tau].results[tau]
    print(
        f"  \u03c4={tau}: \u03bb={res_p.lambda_fe:.4f}, "
        f"# Zero FE = {int(np.sum(np.abs(res_p.fixed_effects) < 1e-6))}"
    )

# Display coefficients
wage_vars = ["const", "exper", "married", "union"]
print(f"\n{'':<12} {'Method':<10}", end="")
for tau in wage_taus:
    print(f"{'\u03c4=' + str(tau):>10}", end="")
print()
print("-" * 72)

for i, var in enumerate(wage_vars):
    # Canay row
    print(f"{var:<12} {'Canay':<10}", end="")
    for tau in wage_taus:
        print(f"{wage_canay[tau].results[tau].params[i]:10.4f}", end="")
    print()

    # Penalty row
    print(f"{'':<12} {'Penalty':<10}", end="")
    for tau in wage_taus:
        print(f"{wage_penalty[tau].results[tau].params[i]:10.4f}", end="")
    print()

In [ ]:
# Analyze worker FE heterogeneity across quantiles
worker_ids = wage_data["id"].unique()

worker_fe = pd.DataFrame(index=worker_ids)
for tau in [0.1, 0.5, 0.9]:
    res = wage_penalty[tau].results[tau]
    model_obj = res.model
    alphas = []
    for wid in worker_ids:
        if wid in model_obj.entity_map:
            idx = model_obj.entity_map[wid]
            alphas.append(res.fixed_effects[idx])
        else:
            alphas.append(0.0)
    worker_fe[f"alpha_{int(100 * tau)}"] = alphas

worker_fe["fe_spread"] = worker_fe["alpha_90"] - worker_fe["alpha_10"]

# Correlations
print("\nWorker FE Correlations Across Quantiles:")
print("=" * 50)
corr_10_50 = worker_fe["alpha_10"].corr(worker_fe["alpha_50"])
corr_50_90 = worker_fe["alpha_50"].corr(worker_fe["alpha_90"])
corr_10_90 = worker_fe["alpha_10"].corr(worker_fe["alpha_90"])
print(f"  \u03b1(0.1) vs \u03b1(0.5): {corr_10_50:.3f}")
print(f"  \u03b1(0.5) vs \u03b1(0.9): {corr_50_90:.3f}")
print(f"  \u03b1(0.1) vs \u03b1(0.9): {corr_10_90:.3f}")

print("\nFE Spread Distribution (\u03b1(0.9) - \u03b1(0.1)):")
print(worker_fe["fe_spread"].describe())

In [ ]:
# Visualization
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Coefficient process - Penalty vs Canay
for i, (var, label) in enumerate([(1, "Experience"), (2, "Married"), (3, "Union")]):
    ax = axes[i]
    canay_coefs = [wage_canay[tau].results[tau].params[var] for tau in wage_taus]
    penalty_coefs = [wage_penalty[tau].results[tau].params[var] for tau in wage_taus]

    ax.plot(
        wage_taus, canay_coefs, "o-", linewidth=2, markersize=7, label="Canay", color="steelblue"
    )
    ax.plot(
        wage_taus, penalty_coefs, "s-", linewidth=2, markersize=7, label="Penalty", color="darkred"
    )
    ax.set_xlabel("Quantile (\u03c4)", fontsize=11)
    ax.set_ylabel("Coefficient", fontsize=11)
    ax.set_title(label, fontsize=13, fontweight="bold")
    ax.legend(fontsize=10)
    ax.grid(alpha=0.3)

plt.suptitle("Wage Equation: Canay vs Penalty FE-QR", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Economic interpretation
print("\n" + "=" * 70)
print("ECONOMIC INTERPRETATION")
print("=" * 70)

print("""
Worker Fixed Effects in Wage Quantile Regression:

The worker FE (alpha_i) captures unobserved time-invariant characteristics
like innate ability, motivation, social networks, and unmeasured skills.

If alpha_i varies across quantiles:
  - A worker with high alpha_i(0.9) but low alpha_i(0.1) has a "high
    ceiling" but uncertain floor -- their ability amplifies good outcomes
    more than it prevents bad outcomes.
  - A worker with similar alpha_i across all tau has a stable contribution
    to their wage regardless of other circumstances.

Policy Implications:
  - If FE vary by quantile, wage inequality is not just about
    observable characteristics (education, experience) but also
    about how unobserved ability interacts with the position in
    the wage distribution.
  - Canay would miss this nuance by forcing alpha_i constant.
  - The penalty method reveals how "ability" operates differently
    at different parts of the wage distribution.
""")

print("All exercises completed!")